# Does bounded confidence change the controversy-axis result?

An A/B backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish),
run entirely against a **local** Ollama model — no API key, $0 — following the
same pattern as `kaggle_controversy.ipynb`, but local rather than Kaggle-hosted
since `qwen2.5:7b` is already pulled on this machine.

## What's being tested

METHODOLOGY.md's calibrated controversy run found the simulation's crowd-split
prediction lands **below chance** (38% vs. a 53% best baseline, n=74). Since
then, T3's herding update gained a bounded-confidence gate (Hegselmann-Krause
style): an agent ignores a herding target more than `confidence_bound` away
from its own opinion, instead of being pulled toward it (damped or not). The
hypothesis was that this produces more realistic multi-modal splits than the
jitter-dependent fragmentation the engine relied on before.

This notebook runs the **same calibrated-controversy harness**, same events,
same split logic, twice — once with the gate on (current default), once with
it forced off (`confidence_bound=2.0` for everyone, i.e. the exact pre-gate
behavior) — and compares. This is a mechanism test, not a fresh claim that
bounded confidence "works": the honest prior, given every other axis on this
domain has failed the ladder, is that it won't move the needle much. Reporting
that plainly either way is the point.

## Why local and not the Anthropic API

No `ANTHROPIC_API_KEY` is configured in the environment this was built in.
Ollama was already running locally with `qwen2.5:7b` pulled, so this uses that
instead — the harness has supported `ollama:<model>` since the original HN
domain build (see `LIGHTNINGFISH_MODEL` in `run_backtest.py`'s docstring).

**This machine has no GPU** (`nvidia-smi` not found) — inference runs on CPU,
measured at roughly 4 tokens/sec generation. A timed probe (8 agents, 2 rounds,
one real story) took ~150s. To keep total runtime to a few hours rather than
the "zero events in 80 minutes" the Kaggle notebook hit on its CPU fallback,
this run uses a **smaller population (10 agents, 3 rounds)** than the
Kaggle-hosted GPU runs (24 agents, 3-4 rounds) elsewhere in this repo. Smaller
n means wider confidence intervals — this is a real backtest, not a toy, but
its power is lower than the GPU-run numbers already in METHODOLOGY.md, and the
result here should be read as directional, not a replacement for a GPU-backed
re-run at the usual size.


---
## 1. Setup

Assumes Ollama is already running locally (`ollama serve`) with `qwen2.5:7b`
pulled (`ollama pull qwen2.5:7b`). No install step — unlike the Kaggle
notebooks, this one runs inside the project's own environment.


In [ ]:
import os, subprocess, sys, time

os.environ["LIGHTNINGFISH_MODEL"] = "ollama:qwen2.5:7b"
os.environ["LIGHTNINGFISH_N_AGENTS"] = "10"
os.environ["LIGHTNINGFISH_N_ROUNDS"] = "3"

r = subprocess.run(["curl", "-s", "http://localhost:11434/api/tags"], capture_output=True, text=True)
assert "qwen2.5:7b" in r.stdout, f"qwen2.5:7b not found in local Ollama — run `ollama pull qwen2.5:7b` first.\n{r.stdout}"
print("Ollama reachable, qwen2.5:7b present.")


---
## 2. Run both arms

`hn-controversy-calibrated` (gate on) and `hn-controversy-calibrated-nobc`
(gate off) share the exact same event pull, calibration/evaluation split
(deterministic hash of event id), and calibration-threshold logic — the run
key only differs by the bounded-confidence flag, so results are comparable.

`limit=250` raw points-sorted stories, pulled once and cached to
`.cache/lightningfish/hn_stories.json` (shared with every other backtest in
this repo). A first attempt at `limit=60` only yielded 20 scoreable events
(33%) — well under the harness's minimum 15/15 calibration/evaluation split —
so this pull is sized with real margin rather than assuming the ~69% rate
seen in a different (older) sample.

Each arm simulates fresh (no local run cache exists yet for this model/size
combination) — expect several hours total for both arms at this size on CPU.
This cell runs to completion; there is no `%%time`-style bail-out because a
partial run isn't a valid calibration split.


In [ ]:
t0 = time.time()
from tests.integration.run_backtest import _run_hn_controversy_calibrated

print("=== bounded confidence ON (current default) ===")
_run_hn_controversy_calibrated(["250"], bounded_confidence=True)
print(f"\n[{time.time() - t0:.0f}s elapsed]")


In [ ]:
print("=== bounded confidence OFF (pre-gate control) ===")
_run_hn_controversy_calibrated(["250"], bounded_confidence=False)
print(f"\n[{time.time() - t0:.0f}s elapsed total]")


---
## 3. Reading the result

Both `_print_report` blocks above report `sim_correct/n`, the majority-class
and naive/single-LLM baseline accuracies, and `p_value_vs_best` (one-sided
binomial test against the best reference — see METHODOLOGY.md's
"Significance" section for why this, not raw accuracy, is what to trust).

Compare the two `sim` accuracies:

- **Materially higher with the gate on** — bounded confidence is worth
  carrying forward as a real mechanism, pending a full-size GPU re-run to
  confirm at proper power.
- **About the same, or worse** — the honest conclusion the rest of this
  project's findings would predict: HN reception is driven by who posts and
  replies early, not by which local opinion-update rule the crowd uses, and
  this axis stays a **fails** either way. Update METHODOLOGY.md's "Worked
  results" table with whichever it is — a negative result here is exactly as
  reportable as every other row in that table.

At n≈20-25 per arm (after the 40/60 calibration/evaluation split), individual
accuracy points move by roughly one event each — do not over-read a few
points of difference; look at whether `beats_baselines` and the verdict
actually flip.
